# All you need to modify

In [1]:
import pandas as pd
import os
import numpy as np

# Input file paths
students_sheets = pd.read_excel(
    '/Users/lgteam/Desktop/Weekly Updates Automation Tool/Data/Weekly Reports (Guo) 0504 - CCM - Students - Oct 30 2026.xlsx',
    sheet_name=None
)

adults_sheets = '/Users/lgteam/Desktop/Weekly Updates Automation Tool/Data/Weekly Reports (Guo) 0504 - CCM - Adults - Oct 30 2026.xlsx'

all_sheets = '/Users/lgteam/Desktop/Weekly Updates Automation Tool/Data/Weekly Reports (Guo) 0504 - CCM - All - Oct 30 2026.xlsx'

annual_schedule_path = '/Users/lgteam/Desktop/Weekly Updates Automation Tool/Data/Activity_Session Details - CCM - Oct 30 2026.xlsx'

missing_attendance_path = "/Users/lgteam/Desktop/Weekly Updates Automation Tool/Data/Gary's Missing Attendance Report - CCM - Oct 30 2026.xlsx"

# Student Summary Statistics - Target # of Students Served (don't include Total)
target_values = [130,190, 255]


/Users/lgteam/Library/Python/3.9/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


# Code Part (don't need to modify)

# Updated info:
1. Created master site list to include sites that have not entered any data to date
2. Included Missing Attendance (%), Staff Assigned, Partner Assigned in the "Site Summary Reports: Attendance by Activity" sheet;
Precaution: 
(a) Partner Assigned col must be manually checked to ensure accuracy as the tool only detects empty entry; 
(b) Color coding must be manually applied for family events, community service learning projects, and summer programming
3. Updadated the adult participation calculation
4. Community Service Summary Statistics needs to be pulled from a separate report
5. Missing Attendance (%) needs to be pulled from a separate report

In [2]:
# Master site list, including sites represented only by placeholder rows.
student_hours = students_sheets["Participant Attendance Hours"].copy()

# Actual headers are in Excel row 3
# (index 1 because read_excel already used row 1 as its header)
student_hours.columns = student_hours.iloc[1]
student_hours = student_hours.iloc[2:].reset_index(drop=True)
student_hours.columns.name = None

# Keep site names, including sites with only placeholder rows
site_names = student_hours["Site"].astype("string").str.strip()

valid_sites = (
    site_names.notna()
    & site_names.ne("")
    & ~site_names.str.lower().isin(["nan", "total", "subtotal"])
)

site_master = (
    site_names.loc[valid_sites]
    .drop_duplicates()
    .rename("Site")
    .to_frame()
    .reset_index(drop=True)
)

if len(site_master) != len(target_values):
    raise ValueError(
        f"Found {len(site_master)} sites but "
        f"{len(target_values)} targets."
    )

site_master["Target # of students served"] = target_values

print(f"Found {len(site_master)} sites. Verify the targets:")
display(site_master)

Found 3 sites. Verify the targets:


,Site,Target # of students served
0,East New York Middle School of Excellence,130
1,P.S. 006 West Farms,190
2,P.S. 181 Brooklyn,255


## 🍀Student Summary Statistics:


In [3]:
# Import the "Participants By Hour Band (Site)" sheet and clean it up
df_part_by_hour = students_sheets[
    "Participants By Hour Band (Site"
].copy()

# Find the first row containing any master-list site
master_sites = site_master["Site"].astype(str).str.strip()

matches = df_part_by_hour.apply(
    lambda col: col.astype(str).str.strip().isin(master_sites)
).any(axis=1)

if not matches.any():
    raise ValueError("No master-list sites found in the hour-band sheet.")

first_site_row = matches[matches].index[0]

if first_site_row == 0:
    raise ValueError("Cannot find a header row above the first site.")

# Use the preceding row as headers, then keep the data rows
df_part_by_hour.columns = (
    df_part_by_hour.iloc[first_site_row - 1]
    .astype("string")
    .str.strip()
)

df_part_by_hour = (
    df_part_by_hour.iloc[first_site_row:]
    .reset_index(drop=True)
)
df_part_by_hour.columns.name = None

if "Site" not in df_part_by_hour.columns:
    raise ValueError("The detected header row does not contain 'Site'.")

display(df_part_by_hour.head(10))

,Institution,Site,Registered Adults,Attended Adults,Registered Youth,Attended Youth,Less Than 15 Hours,15-44 Hours,45-89 Hours,90-179 Hours,180-269 Hours,270+ Hours,15+ Hours,Last Attendance Date
0,CCMS - Community Counseling and Mediation 8071,East New York Middle School of Excellence,0,0,1,0,0,0,0,0,0,0,0,NaN
1,CCMS - Community Counseling and Mediation 8071,P.S. 006 West Farms,0,0,184,0,0,0,0,0,0,0,0,NaN
2,CCMS - Community Counseling and Mediation 8071,P.S. 181 Brooklyn,1,0,93,90,2,29,59,0,0,0,88,2026-10-29 00:00:00
3,NaN,NaN,1,0,278,90,2,29,59,0,0,0,88,NaN


In [4]:
# Keep actual site rows, regardless of Institution
site_names = df_part_by_hour["Site"].astype("string").str.strip()

valid_site = (
    site_names.notna()
    & site_names.ne("")
    & ~site_names.str.lower().isin(["nan", "subtotal", "total"])
)

df_part_by_hour = df_part_by_hour.loc[valid_site].copy()
df_part_by_hour["Site"] = site_names.loc[valid_site]
df_part_by_hour = df_part_by_hour.reset_index(drop=True)

df_part_by_hour

,Institution,Site,Registered Adults,Attended Adults,Registered Youth,Attended Youth,Less Than 15 Hours,15-44 Hours,45-89 Hours,90-179 Hours,180-269 Hours,270+ Hours,15+ Hours,Last Attendance Date
0,CCMS - Community Counseling and Mediation 8071,East New York Middle School of Excellence,0,0,1,0,0,0,0,0,0,0,0,NaN
1,CCMS - Community Counseling and Mediation 8071,P.S. 006 West Farms,0,0,184,0,0,0,0,0,0,0,0,NaN
2,CCMS - Community Counseling and Mediation 8071,P.S. 181 Brooklyn,1,0,93,90,2,29,59,0,0,0,88,2026-10-29 00:00:00


In [5]:
# Get the average  column from Daily Site Attendance Summary
daily_site_att = students_sheets[
    "Daily Site Attendance Summary"
].copy()

master_sites = site_master["Site"].astype(str).str.strip()

matches = daily_site_att.apply(
    lambda col: col.astype(str).str.strip().isin(master_sites)
).any(axis=1)

if not matches.any():
    # No site attendance data: retain an empty table for merging later
    daily_site_att = pd.DataFrame(columns=["Site"])
else:
    first_site_row = matches[matches].index[0]

    if first_site_row == 0:
        raise ValueError("Cannot find a header row above daily attendance.")

    daily_site_att.columns = daily_site_att.iloc[first_site_row - 1]
    daily_site_att = (
        daily_site_att.iloc[first_site_row:]
        .reset_index(drop=True)
    )
    daily_site_att.columns.name = None

    daily_site_att["Site"] = (
        daily_site_att["Site"].astype("string").str.strip()
    )

    daily_site_att = daily_site_att.loc[
        daily_site_att["Site"].isin(master_sites)
    ].copy()

    if daily_site_att["Site"].duplicated().any():
        raise ValueError("Duplicate sites found in daily attendance.")

display(daily_site_att.head())

,Site,2026-10-05 00:00:00,2026-10-06 00:00:00,2026-10-07 00:00:00,2026-10-08 00:00:00,2026-10-09 00:00:00,2026-10-13 00:00:00,2026-10-14 00:00:00,2026-10-15 00:00:00,2026-10-16 00:00:00,2026-10-19 00:00:00,2026-10-20 00:00:00,2026-10-21 00:00:00,2026-10-22 00:00:00,2026-10-23 00:00:00,2026-10-26 00:00:00,2026-10-27 00:00:00,2026-10-28 00:00:00,2026-10-29 00:00:00
0,P.S. 181 Brooklyn,68,68,71,72,81,77,80,73,72,72,72,75,66,69,70,73,73,64


In [6]:
# Calculate the average # of students served per site across the date cols, rounding to the nearest whole number

from datetime import datetime, date

date_cols = [
    col for col in daily_site_att.columns
    if isinstance(col, (pd.Timestamp, datetime, date))
]

if daily_site_att.empty:
    daily_site_att["Total"] = pd.Series(dtype="Int64")
else:
    if not date_cols:
        raise ValueError("No date columns found in daily attendance.")

    daily_values = daily_site_att[date_cols].apply(
        pd.to_numeric,
        errors="coerce"
    )

    daily_site_att["Total"] = (
        daily_values.mean(axis=1).round().astype("Int64")
    )

daily_site_att = daily_site_att[["Site", "Total"]]

display(daily_site_att)

,Site,Total
0,P.S. 181 Brooklyn,72


In [7]:
print(df_part_by_hour.columns)

Index(['Institution', 'Site', 'Registered Adults', 'Attended Adults',
       'Registered Youth', 'Attended Youth', 'Less Than 15 Hours',
       '15-44 Hours', '45-89 Hours', '90-179 Hours', '180-269 Hours',
       '270+ Hours', '15+ Hours', 'Last Attendance Date'],
      dtype='string')


In [8]:
all_cols = ['0 Hours', 'Less Than 15 Hours', '15-44 Hours', '45-89 Hours', '90-179 Hours', '180-269 Hours', '270+ Hours']
served_cols = ['Less Than 15 Hours', '15-44 Hours', '45-89 Hours', '90-179 Hours', '180-269 Hours', '270+ Hours']
plus15_cols = ['15-44 Hours', '45-89 Hours', '90-179 Hours', '180-269 Hours', '270+ Hours']
plus90_cols = ['90-179 Hours', '180-269 Hours', '270+ Hours']

# Filter each col list to only include columns that exist in the dataframe
existing_cols = df_part_by_hour.columns.tolist()
all_cols     = [c for c in all_cols     if c in existing_cols]
served_cols  = [c for c in served_cols  if c in existing_cols]
plus15_cols  = [c for c in plus15_cols  if c in existing_cols]
plus90_cols  = [c for c in plus90_cols  if c in existing_cols]

# Convert hour-band counts to numeric
df_part_by_hour[all_cols] = df_part_by_hour[all_cols].apply(
    pd.to_numeric, errors="coerce"
).fillna(0)

# Enrollment includes students who have not attended
df_part_by_hour["Registered Youth"] = pd.to_numeric(
    df_part_by_hour["Registered Youth"],
    errors="coerce"
).fillna(0)

# Build summary with Site retained for matching
df_totals = pd.DataFrame({
    "Site": df_part_by_hour["Site"],
    "Total # Enrolled": df_part_by_hour["Registered Youth"],
    "Total # Served": df_part_by_hour[served_cols].sum(axis=1),
    "Total # 15+": df_part_by_hour[plus15_cols].sum(axis=1),
    "Total # 90+": df_part_by_hour[plus90_cols].sum(axis=1),
})

# Match targets by site rather than row position
df_totals = site_master[
    ["Site", "Target # of students served"]
].merge(
    df_totals,
    on="Site",
    how="left",
    validate="one_to_one"
)

display(df_totals)

,Site,Target # of students served,Total # Enrolled,Total # Served,Total # 15+,Total # 90+
0,East New York Middle School of Excellence,130,1,0,0,0
1,P.S. 006 West Farms,190,184,0,0,0
2,P.S. 181 Brooklyn,255,93,90,88,0


In [9]:
## Add average daily attendance by Site
daily_site_att["Total"] = daily_site_att["Total"].astype("Int64")

daily_site_att = daily_site_att.rename(
    columns={"Total": "Avg. # of Students Per Day"}
)

df_totals = df_totals.merge(
    daily_site_att[["Site", "Avg. # of Students Per Day"]],
    on="Site",
    how="left",
    validate="one_to_one"
)

display(df_totals)

,Site,Target # of students served,Total # Enrolled,Total # Served,Total # 15+,Total # 90+,Avg. # of Students Per Day
0,East New York Middle School of Excellence,130,1,0,0,0,<NA>
1,P.S. 006 West Farms,190,184,0,0,0,<NA>
2,P.S. 181 Brooklyn,255,93,90,88,0,72


In [10]:
## School names and total row
df_totals = df_totals.rename(columns={"Site": "School"})

df_totals = df_totals[
    [
        "School",
        "Target # of students served",
        "Total # Enrolled",
        "Total # Served",
        "Avg. # of Students Per Day",
        "Total # 15+",
        "Total # 90+",
    ]
]

# Sum numeric columns
total_row = df_totals.iloc[:, 1:].sum(min_count=1).to_frame().T
total_row.insert(0, "School", "Total")

# Avoid presenting a partial attendance sum as the complete grant total
if df_totals["Avg. # of Students Per Day"].isna().any():
    total_row["Avg. # of Students Per Day"] = pd.NA

df_totals = pd.concat(
    [df_totals, total_row],
    ignore_index=True
)

display(df_totals)

/var/folders/jz/3h0z6zcs4rs6s0169f_848f40000gn/T/ipykernel_98026/4063669675.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_totals = pd.concat(


,School,Target # of students served,Total # Enrolled,Total # Served,Avg. # of Students Per Day,Total # 15+,Total # 90+
0,East New York Middle School of Excellence,130,1,0,<NA>,0,0
1,P.S. 006 West Farms,190,184,0,<NA>,0,0
2,P.S. 181 Brooklyn,255,93,90,72,88,0
3,Total,575,278,90,<NA>,88,0


In [11]:
# Format counts and percentages
for hours in [15, 90]:
    count_col = f"Total # {hours}+"
    display_col = f"# of students {hours}+ hrs total (% of Target)"

    if count_col in df_totals.columns:
        counts = pd.to_numeric(
            df_totals[count_col], errors="coerce"
        ).astype("Int64")

        targets = pd.to_numeric(
            df_totals["Target # of students served"],
            errors="coerce"
        ).astype("Float64")

        percentages = (
            counts.div(targets.where(targets > 0))
            .mul(100)
            .round()
            .astype("Int64")
        )

        df_totals[display_col] = (
            counts.astype("string").fillna("-")
            + " ("
            + percentages.astype("string").fillna("-")
            + "%)"
        )

        df_totals.loc[counts.isna(), display_col] = "-"

    elif display_col not in df_totals.columns:
        raise ValueError(
            f"Missing {count_col}. Rerun the student-count cells first."
        )

# Remove raw count columns if present
df_totals = df_totals.drop(
    columns=["Total # 15+", "Total # 90+"],
    errors="ignore"
).reset_index(drop=True)

# Keep numeric counts as integers, allowing missing values
count_columns = [
    "Target # of students served",
    "Total # Enrolled",
    "Total # Served",
    "Avg. # of Students Per Day",
]

for col in count_columns:
    df_totals[col] = pd.to_numeric(
        df_totals[col], errors="coerce"
    ).astype("Int64")


# Color valid 15+ percentages
def color_pct_cols(val):
    try:
        pct = int(str(val).split("(")[1].replace("%)", "").strip())
        return "color: green" if pct >= 100 else "color: red"
    except (ValueError, IndexError):
        return ""


styled_totals = (
    df_totals.style
    .apply(
        lambda col: col.map(color_pct_cols),
        subset=["# of students 15+ hrs total (% of Target)"]
    )
    .format(
        {col: "{:.0f}" for col in count_columns},
        na_rep="-"
    )
)

display(styled_totals)

,School,Target # of students served,Total # Enrolled,Total # Served,Avg. # of Students Per Day,# of students 15+ hrs total (% of Target),# of students 90+ hrs total (% of Target)
0,East New York Middle School of Excellence,130,1,0,-,0 (0%),0 (0%)
1,P.S. 006 West Farms,190,184,0,-,0 (0%),0 (0%)
2,P.S. 181 Brooklyn,255,93,90,72,88 (35%),0 (0%)
3,Total,575,278,90,-,88 (15%),0 (0%)


## 🌷Family Component Summary Statistics (Last Column)

In [12]:
df_hours = pd.read_excel(
    adults_sheets,
    sheet_name="Participant Attendance Hours",
    header=None
)

header_text = df_hours.apply(
    lambda col: col.astype("string").str.strip()
)

header_matches = (
    header_text.eq("Site").any(axis=1)
    & header_text.eq("Participant Id").any(axis=1)
    & header_text.eq("Hours Present").any(axis=1)
)

if not header_matches.any():
    raise ValueError(
        "Cannot find Site, Participant Id, and Hours Present headers."
    )

header_row = header_matches[header_matches].index[0]

df_hours.columns = header_text.iloc[header_row]
df_hours = df_hours.iloc[header_row + 1:].reset_index(drop=True)
df_hours.columns.name = None

print(f"Header detected in Excel row {header_row + 1}.")
display(df_hours.head())

Header detected in Excel row 3.


/Users/lgteam/Library/Python/3.9/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Institution,Site,Last Name,First Name,Participant Id,State Participant Id,School,Grade Level,Teachers,Days Present,Hours Present,Hour Band
0,CCMS - Community Counseling and Mediation 8071,P.S. 181 Brooklyn,Umoja,Thandi,1792607484555125TU,1792607484555125TU,P.S. 181 Brooklyn,AE,NaN,0,0,0


In [13]:
df_hours["Site"] = (
    df_hours["Site"].astype("string").str.strip()
)

df_hours["Hours Present"] = pd.to_numeric(
    df_hours["Hours Present"], errors="coerce"
)

df_hours["Participant Id"] = (
    df_hours["Participant Id"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

valid_id = (
    df_hours["Participant Id"].notna()
    & df_hours["Participant Id"].ne("")
    & ~df_hours["Participant Id"].str.lower().isin(["nan", "none"])
)

df_active = df_hours.loc[
    (df_hours["Hours Present"] > 0)
    & valid_id
    & df_hours["Participant Id"].str.len().ne(9)
].copy()

# Check that qualifying participants have recognized sites
if df_active["Site"].isna().any() or df_active["Site"].eq("").any():
    raise ValueError("An adult with attendance has no Site.")

unexpected_sites = (
    set(df_active["Site"]) - set(site_master["Site"])
)

if unexpected_sites:
    raise ValueError(
        f"Adult sites missing from master list: {unexpected_sites}"
    )

print(f"Qualifying attendance records: {len(df_active)}")

Qualifying attendance records: 0


In [14]:
result = (
    df_active.groupby("Site")["Participant Id"]
    .nunique()
    .reset_index(name="Parents Served (Total)")
)

result = site_master[["Site"]].merge(
    result,
    on="Site",
    how="left",
    validate="one_to_one"
)

result["Parents Served (Total)"] = (
    result["Parents Served (Total)"].fillna(0).astype(int)
)

result.loc[len(result)] = {
    "Site": "Total",
    "Parents Served (Total)": result["Parents Served (Total)"].sum()
}

display(result)

,Site,Parents Served (Total)
0,East New York Middle School of Excellence,0
1,P.S. 006 West Farms,0
2,P.S. 181 Brooklyn,0
3,Total,0


##  🌸Participant Demographics - missing info

In [15]:
# Load the Participant Demographics sheet from the students workbook and clean it up
df_part_demo = students_sheets["Participant Demographics"].copy()

# If headers are already correct, no detection is needed
if not {"Site", "Date Of Birth", "Grade Level"}.issubset(
    df_part_demo.columns
):
    header_text = df_part_demo.apply(
        lambda col: col.astype("string").str.strip()
    )

    header_matches = (
        header_text.eq("Site").any(axis=1)
        & header_text.eq("Date Of Birth").any(axis=1)
        & header_text.eq("Grade Level").any(axis=1)
    )

    if not header_matches.any():
        raise ValueError(
            "Cannot find demographics headers. "
            "Rerun the original import cell to reload students_sheets."
        )

    header_row = header_matches[header_matches].index[0]

    df_part_demo.columns = header_text.iloc[header_row]
    df_part_demo = (
        df_part_demo.iloc[header_row + 1:]
        .reset_index(drop=True)
    )

df_part_demo.columns = df_part_demo.columns.astype(str).str.strip()
df_part_demo.columns.name = None

# Validate expected columns
required_columns = [
    "Site",
    "Last Name",
    "First Name",
    "ParticipantID",
    "State ParticipantID",
    "Date Of Birth",
    "Grade Level",
    "Gender",
]

missing_columns = [
    col for col in required_columns
    if col not in df_part_demo.columns
]

if missing_columns:
    raise ValueError(
        f"Missing demographics columns: {missing_columns}"
    )

# Exclude empty placeholders; retain students whose ID is missing
identity_columns = ["Last Name", "First Name", "ParticipantID"]

has_identity = df_part_demo[identity_columns].apply(
    lambda col: (
        col.astype("string").str.strip().fillna("").ne("")
    )
).any(axis=1)

df_part_demo = df_part_demo.loc[has_identity].reset_index(drop=True)

print(f"Participant records loaded: {len(df_part_demo)}")
display(df_part_demo.head())

Participant records loaded: 278


,Institution,Site,Last Name,First Name,ParticipantID,State ParticipantID,Adult,Registration Start Date,Registration End Date,Currently Active,Date Of Birth,LEA,School,Grade Level,Gender,Race/Ethnicity,English Learner Status,Lunch Status,Special Education Status,IDEA Disability Type
0,CCMS - Community Counseling and Mediation 8071,P.S. 006 West Farms,Abdulaziz,Labib,260026661,2377386810,No,2026-07-01,2027-06-30,Yes,2018-02-17,"NYC Community School District 12, 8071",P.S. 006 West Farms,03,Male,Black or African American,No,Free,No,Not Entered
1,CCMS - Community Counseling and Mediation 8071,P.S. 006 West Farms,Abreu Pichardo,Christopher,250356912,6038522641,No,2026-07-01,2027-06-30,Yes,2016-01-06,"NYC Community School District 12, 8071",P.S. 006 West Farms,05,Male,Race and Ethnicity Unknown,No,Not Entered,No,Not Entered
2,CCMS - Community Counseling and Mediation 8071,P.S. 006 West Farms,Abril,Leilani,260728019,8790208944,No,2026-07-01,2027-06-30,Yes,2018-03-20,"NYC Community School District 12, 8071",P.S. 006 West Farms,03,Female,Hispanic or Latino,No,Free,No,Not Entered
3,CCMS - Community Counseling and Mediation 8071,P.S. 006 West Farms,Abril,Malea,250029612,1636288514,No,2026-07-01,2027-06-30,Yes,2019-06-26,"NYC Community School District 12, 8071",P.S. 006 West Farms,02,Female,Hispanic or Latino,No,Free,No,Not Entered
4,CCMS - Community Counseling and Mediation 8071,P.S. 006 West Farms,Acevedo,Esthela,259773786,2249194876,No,2026-07-01,2027-06-30,Yes,2020-04-11,"NYC Community School District 12, 8071",P.S. 006 West Farms,01,Female,Hispanic or Latino,No,Free,No,Not Entered


In [16]:
# Flag missing required demo info
all_missing_flags = df_part_demo.copy()

def clean_text(series):
    return series.astype("string").str.strip()

def is_missing(series):
    text = clean_text(series)
    return (
        text.isna()
        | text.str.lower().isin(
            ["", "not entered", "nan", "none"]
        )
    )

# Clean site names
all_missing_flags["Site"] = clean_text(all_missing_flags["Site"])
all_missing_flags.loc[
    is_missing(all_missing_flags["Site"]), "Site"
] = pd.NA

# Clean IDs
for col in ["ParticipantID", "State ParticipantID"]:
    all_missing_flags[col] = (
        clean_text(all_missing_flags[col])
        .str.replace(r"\.0$", "", regex=True)
    )

# Missing or unparseable DOB
dob_parsed = pd.to_datetime(
    all_missing_flags["Date Of Birth"],
    errors="coerce"
)

all_missing_flags["Date Of Birth_missing"] = (
    dob_parsed.isna().astype(int)
)

# Required participant ID: nine digits starting with 1 or 2
all_missing_flags["ParticipantID_missing"] = (
    ~all_missing_flags["ParticipantID"].str.fullmatch(
        r"[12]\d{8}", na=False
    )
).astype(int)

all_missing_flags["Grade Level_missing"] = (
    is_missing(all_missing_flags["Grade Level"]).astype(int)
)

all_missing_flags["Gender_missing"] = (
    ~clean_text(all_missing_flags["Gender"])
    .str.lower()
    .isin(["male", "female", "non-binary"])
).astype(int)

# Optional state ID: ten digits
all_missing_flags["State ParticipantID_missing"] = (
    ~all_missing_flags["State ParticipantID"].str.fullmatch(
        r"\d{10}", na=False
    )
).astype(int)

In [17]:
# Flag unusual age for grade
# Allow for students who are up to 3 years older or younger than typical for their grade;
# Allow for students 1-9 years old for PK/KG

report_date = pd.Timestamp.today().normalize()
age_tolerance = 3

grade_text = clean_text(
    all_missing_flags["Grade Level"]
).str.upper()

grade_number = pd.to_numeric(grade_text, errors="coerce")

# Recognize PK and kindergarten labels
is_pk_kg = grade_text.isin([
    "PK", "PRE-K", "PREK", "PRE-KINDERGARTEN",
    "K", "KG", "KN", "KINDERGARTEN"
]) | grade_number.eq(0)

# Apply typical-age screening only to Grades 1–12
recognized_grade = (
    grade_number.between(1, 12)
    & grade_number.mod(1).eq(0)
).fillna(False)

typical_age = (grade_number + 5).where(recognized_grade)

# Age in completed years
birthday_not_reached = (
    (dob_parsed.dt.month > report_date.month)
    | (
        (dob_parsed.dt.month == report_date.month)
        & (dob_parsed.dt.day > report_date.day)
    )
)

age = (
    report_date.year
    - dob_parsed.dt.year
    - birthday_not_reached.astype(int)
)

# PK/KG: allow ages 1–9 inclusive
pk_kg_mismatch = (
    is_pk_kg
    & dob_parsed.notna()
    & (age.lt(1) | age.gt(9))
).fillna(False)

# Grades 1–12: allow typical age ±3 inclusive
age_grade_mismatch = (
    recognized_grade
    & dob_parsed.notna()
    & (age - typical_age).abs().gt(age_tolerance)
).fillna(False)

future_dob = dob_parsed.gt(report_date)

all_missing_flags["DOB_age_grade_review"] = (
    pk_kg_mismatch | age_grade_mismatch | future_dob
).astype(int)

dob_review_details = pd.DataFrame({
    "Age at Report Date": age.astype("Int64"),
    "Typical Age for Grade": typical_age.astype("Int64"),
    "Review Reason": np.select(
        [
            future_dob.to_numpy(dtype=bool),
            pk_kg_mismatch.to_numpy(dtype=bool),
            age_grade_mismatch.to_numpy(dtype=bool),
        ],
        [
            "DOB is in the future",
            "PK/KG age outside 1–9 years",
            "Age more than 3 years from typical grade age",
        ],
        default=""
    ),
})

unrecognized_grades = grade_text.loc[
    ~recognized_grade
    & ~is_pk_kg
    & ~is_missing(all_missing_flags["Grade Level"])
].drop_duplicates().tolist()

print("Grades not included in age screening:", unrecognized_grades)
print(
    "DOB review records:",
    all_missing_flags["DOB_age_grade_review"].sum()
)

Grades not included in age screening: []
DOB review records: 2


In [18]:
# Create Missing Student Summary
summary_flags = [
    "Date Of Birth_missing",
    "ParticipantID_missing",
    "Grade Level_missing",
    "Gender_missing",
    "State ParticipantID_missing",
]

unexpected_sites = (
    set(all_missing_flags["Site"].dropna())
    - set(site_master["Site"])
)

if unexpected_sites:
    raise ValueError(
        f"Demographics sites missing from master list: {unexpected_sites}"
    )

missing_summary = (
    all_missing_flags.groupby("Site")[summary_flags]
    .sum()
    .reset_index()
)

missing_summary = site_master[["Site"]].merge(
    missing_summary,
    on="Site",
    how="left",
    validate="one_to_one"
)

missing_summary[summary_flags] = (
    missing_summary[summary_flags].fillna(0).astype(int)
)

total_row = {"Site": "Total"}
total_row.update(missing_summary[summary_flags].sum().to_dict())

missing_summary = pd.concat(
    [missing_summary, pd.DataFrame([total_row])],
    ignore_index=True
)

missing_summary = missing_summary.rename(columns={
    "Date Of Birth_missing": "DOB_missing",
    "State ParticipantID_missing":
        "10digit_State ParticipantID_missing",
})

display(missing_summary)

,Site,DOB_missing,ParticipantID_missing,Grade Level_missing,Gender_missing,10digit_State ParticipantID_missing
0,East New York Middle School of Excellence,0,1,0,0,1
1,P.S. 006 West Farms,0,3,0,0,5
2,P.S. 181 Brooklyn,0,0,0,0,30
3,Total,0,4,0,0,36


In [19]:
# Create student pull-outs
flag_cols = [
    "Date Of Birth_missing",
    "ParticipantID_missing",
    "Grade Level_missing",
    "Gender_missing",
]

total_missing_rows = all_missing_flags.loc[
    all_missing_flags[flag_cols].sum(axis=1) > 0
].copy()

missing_site_rows = all_missing_flags.loc[
    all_missing_flags["Site"].isna()
].copy()

review_mask = all_missing_flags["DOB_age_grade_review"].eq(1)

young_dob_rows = all_missing_flags.loc[review_mask].copy()

young_dob_rows = young_dob_rows.join(
    dob_review_details.loc[review_mask]
)

print(f"Required-info review records: {len(total_missing_rows)}")
print(f"Records without a site: {len(missing_site_rows)}")
print(f"DOB/grade review records: {len(young_dob_rows)}")

Required-info review records: 4
Records without a site: 0
DOB/grade review records: 2


## Missing Staff Summary

In [20]:
# Load the Staff Details sheet again to detect the header row and validate required columns

df_staff = pd.read_excel(
    all_sheets,
    sheet_name="Staff Details",
    header=None
)

header_text = df_staff.apply(
    lambda col: col.astype("string").str.strip()
)

header_matches = (
    header_text.eq("Site").any(axis=1)
    & header_text.eq("Staff Type").any(axis=1)
    & header_text.eq("Compensation Type").any(axis=1)
)

if not header_matches.any():
    raise ValueError("Cannot find the Staff Details header row.")

header_row = header_matches[header_matches].index[0]

df_staff.columns = header_text.iloc[header_row]
df_staff = df_staff.iloc[header_row + 1:].reset_index(drop=True)
df_staff.columns.name = None

required_columns = [
    "Site", "Email Address", "First Name", "Last Name",
    "Staff Type", "Compensation Type", "Funder",
]

missing_columns = [
    col for col in required_columns
    if col not in df_staff.columns
]

if missing_columns:
    raise ValueError(f"Missing staff columns: {missing_columns}")

print(f"Header detected in Excel row {header_row + 1}.")

/Users/lgteam/Library/Python/3.9/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Header detected in Excel row 3.


In [21]:
df = df_staff.copy()

# Normalize text used in the checks
text_columns = [
    "Site", "Email Address", "First Name", "Last Name",
    "Staff Type", "Compensation Type", "Funder",
]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip().fillna("")
    df.loc[
        df[col].str.lower().isin(["nan", "none", "not entered"]),
        col
    ] = ""

# Exclude site placeholders without any staff identity
has_identity = df[
    ["Email Address", "First Name", "Last Name"]
].ne("").any(axis=1)

df = df.loc[has_identity].copy()

if df["Site"].eq("").any():
    raise ValueError("A staff record has no Site assigned.")

unexpected_sites = set(df["Site"]) - set(site_master["Site"])

if unexpected_sites:
    raise ValueError(
        f"Staff sites missing from master list: {unexpected_sites}"
    )

# Email preferred; otherwise use first and last name
email = df["Email Address"].str.lower()
name_id = (
    df["First Name"].str.lower()
    + "|"
    + df["Last Name"].str.lower()
)

df["_staff_id"] = np.where(
    email.ne(""),
    "email:" + email,
    "name:" + name_id
)

# Preserve the original rule: retain the first record per staff/site
df = df.drop_duplicates(subset=["Site", "_staff_id"]).copy()

staff_type = df["Staff Type"].str.lower()
comp_type = df["Compensation Type"].str.lower()
funder = df["Funder"]

df["Staff Type_missing"] = (
    staff_type.isin(["", "other"])
).astype(int)

df["Employment Type_missing"] = (
    ~comp_type.isin(["paid", "volunteer"])
).astype(int)

# Preserve existing funding rule:
# non-volunteers need a funder; any entered funder must contain "21"
is_volunteer = comp_type.eq("volunteer")
funder_blank = funder.eq("")
funder_not_21cclc = ~funder.str.contains("21", case=False, na=False)

df["Funded by 21st CCLC_missing"] = (
    (~is_volunteer & funder_blank)
    | (~funder_blank & funder_not_21cclc)
).astype(int)

staff_flag_cols = [
    "Staff Type_missing",
    "Employment Type_missing",
    "Funded by 21st CCLC_missing",
]

staff_missing_rows = df.loc[
    df[staff_flag_cols].sum(axis=1) > 0
].copy()

In [22]:
# Create a summary of missing staff info by site
staff_missing_summary = (
    df.groupby("Site")
    .agg(
        **{
            "# of Active Program Staff": ("_staff_id", "nunique"),
            "Staff Type": ("Staff Type_missing", "sum"),
            "Employment Type": ("Employment Type_missing", "sum"),
            "Funded by 21st CCLC": (
                "Funded by 21st CCLC_missing", "sum"
            ),
        }
    )
    .reset_index()
)

staff_missing_summary = site_master[["Site"]].merge(
    staff_missing_summary,
    on="Site",
    how="left",
    validate="one_to_one"
)

summary_columns = [
    "# of Active Program Staff",
    "Staff Type",
    "Employment Type",
    "Funded by 21st CCLC",
]

staff_missing_summary[summary_columns] = (
    staff_missing_summary[summary_columns].fillna(0).astype(int)
)

total_row = {"Site": "Total"}
total_row.update(
    staff_missing_summary[summary_columns].sum().to_dict()
)

staff_missing_summary = pd.concat(
    [staff_missing_summary, pd.DataFrame([total_row])],
    ignore_index=True
)

display(staff_missing_summary)

,Site,# of Active Program Staff,Staff Type,Employment Type,Funded by 21st CCLC
0,East New York Middle School of Excellence,19,0,0,0
1,P.S. 006 West Farms,19,0,0,0
2,P.S. 181 Brooklyn,31,0,0,0
3,Total,69,0,0,0


## 🪻Site Summary Report

In [23]:
# 1. Load session reports and detect their header rows
def read_session_report(sheet_name):
    raw = pd.read_excel(
        all_sheets,
        sheet_name=sheet_name,
        header=None
    )

    header_text = raw.apply(
        lambda col: col.astype("string").str.strip()
    )

    header_matches = (
        header_text.eq("Site").any(axis=1)
        & header_text.eq("Activity").any(axis=1)
        & header_text.eq("Session").any(axis=1)
    )

    if not header_matches.any():
        raise ValueError(
            f"Cannot find Site, Activity, and Session headers "
            f"in '{sheet_name}'."
        )

    header_row = header_matches[header_matches].index[0]

    # Preserve original date headers for the attendance report
    headers = raw.iloc[header_row].map(
        lambda value: value.strip()
        if isinstance(value, str) else value
    )

    df = raw.iloc[header_row + 1:].copy()
    df.columns = headers
    df.columns.name = None

    # Remove columns without a header
    df = df.loc[:, df.columns.notna() & (df.columns != "")]

    if df.columns.duplicated().any():
        raise ValueError(f"Duplicate headers in '{sheet_name}'.")

    # Clean the keys used to match the reports
    for col in ["Site", "Activity", "Session"]:
        df[col] = df[col].astype("string").str.strip()

    # Remove blank rows and report totals without a named site
    df = df.loc[
        df["Site"].notna()
        & df["Site"].ne("")
        & ~df["Site"].str.lower().isin(["total", "subtotal", "average"])
    ].copy()

    unexpected_sites = set(df["Site"]) - set(site_master["Site"])

    if unexpected_sites:
        raise ValueError(
            f"Sites missing from master list in '{sheet_name}': "
            f"{unexpected_sites}"
        )

    return df.reset_index(drop=True)


df_act = read_session_report("Activity-Session Details")
df_enr = read_session_report("Session Enrollment by Session")
df_att = read_session_report("Daily Activity Attendance Summa")

print("Activity/session rows:", len(df_act))
print("Enrollment rows:", len(df_enr))
print("Attendance rows:", len(df_att))

print("\nActivity/session columns:")
print(df_act.columns.tolist())

Activity/session rows: 25
Enrollment rows: 25
Attendance rows: 25

Activity/session columns:
['Institution', 'Site', 'Activity', 'Activity Description', 'Session', 'Session Description', 'Funders', 'Partners', 'Location', 'Designated Staff', 'Designated Staff by Type', 'APR Activity Category', 'Schedule Type', 'Session Start Date', 'Session End Date', 'Session Start Time', 'Session End Time', 'Scheduled DOW', 'Enrolled Count', 'Maximum Capacity', 'Days Scheduled', 'Hours Scheduled']


/Users/lgteam/Library/Python/3.9/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/lgteam/Library/Python/3.9/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/lgteam/Library/Python/3.9/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [24]:
# 2. Prepare Activity-Session Details from the regular grant workbook
cols_act = [
    "Site",
    "Activity",
    "Session",
    "Days Scheduled",
    "Session Start Date",
    "Designated Staff",
    "Partners",
]

df_act = df_act[cols_act].copy()

df_act = df_act.rename(columns={
    "Days Scheduled": "Days Scheduled to Date",
    "Designated Staff": "Staff Assigned",
    "Partners": "Partner",
})

df_act["Days Scheduled to Date"] = pd.to_numeric(
    df_act["Days Scheduled to Date"],
    errors="coerce"
).astype("Int64")

df_act["Session Start Date"] = pd.to_datetime(
    df_act["Session Start Date"],
    errors="coerce"
)

# Normalize blank assignments
for col in ["Staff Assigned", "Partner"]:
    df_act[col] = df_act[col].astype("string").str.strip()

    df_act.loc[
        df_act[col].str.lower().isin(
            ["", "not entered", "nan", "none"]
        ),
        col
    ] = pd.NA

df_act["Partner Assigned"] = np.where(
    df_act["Partner"].notna(), "Y", "N"
)

df_act["Staff Assigned"] = np.where(
    df_act["Staff Assigned"].notna(), "Y", "N"
)

today = pd.Timestamp.today().normalize()

display(df_act.head())

,Site,Activity,Session,Days Scheduled to Date,Session Start Date,Staff Assigned,Partner,Partner Assigned
0,P.S. 181 Brooklyn,Arts & Crafts,Arts & Crafts 1,3,2026-10-05,N,<NA>,N
1,P.S. 181 Brooklyn,Arts & Crafts,Arts & Crafts 2,4,2026-10-05,Y,CCM,Y
2,P.S. 181 Brooklyn,Arts & Crafts,Arts & Crafts K,3,2026-10-05,N,<NA>,N
3,P.S. 181 Brooklyn,Basketball,Basketball,7,2026-10-05,N,<NA>,N
4,P.S. 181 Brooklyn,Culture Club 4-8th/SEL,Culture Club/SEL,11,2026-10-05,Y,CCM,Y


In [25]:
# 3. Extract and format Session Enrollment by Session
cols_enr = ["Site", "Activity", "Session", "Enrolled Count"]

df_enr = df_enr[cols_enr].copy()

df_enr["Enrolled Count"] = pd.to_numeric(
    df_enr["Enrolled Count"],
    errors="coerce"
)

df_enr = df_enr.rename(
    columns={"Enrolled Count": "Enrolled Participant"}
)

display(df_enr.head())

,Site,Activity,Session,Enrolled Participant
0,P.S. 181 Brooklyn,Arts & Crafts,Arts & Crafts 1,9
1,P.S. 181 Brooklyn,Arts & Crafts,Arts & Crafts 2,16
2,P.S. 181 Brooklyn,Arts & Crafts,Arts & Crafts K,11
3,P.S. 181 Brooklyn,Basketball,Basketball,30
4,P.S. 181 Brooklyn,Culture Club 4-8th/SEL,Culture Club/SEL,21


In [26]:
# 4. Extract and format Daily Activity Attendance Summary
cols_att = ["Site", "Activity", "Session", "Total"]

df_att = df_att[cols_att].copy()

# Handle both numeric values and text such as "Average: 12.5"
attendance_text = (
    df_att["Total"]
    .astype("string")
    .str.replace(r"(?i)^average\s*:\s*", "", regex=True)
    .str.strip()
)

df_att["Total"] = (
    pd.to_numeric(attendance_text, errors="coerce")
    .round()
    .astype("Int64")
)

df_att = df_att.rename(
    columns={"Total": "Average Daily Attendance"}
)

display(df_att.head())

,Site,Activity,Session,Average Daily Attendance
0,P.S. 181 Brooklyn,Arts & Crafts,Arts & Crafts 1,8
1,P.S. 181 Brooklyn,Arts & Crafts,Arts & Crafts 2,15
2,P.S. 181 Brooklyn,Arts & Crafts,Arts & Crafts K,8
3,P.S. 181 Brooklyn,Basketball,Basketball,21
4,P.S. 181 Brooklyn,Culture Club 4-8th/SEL,Culture Club/SEL,14


In [27]:
# 5. Load the separate full-year activity details report for the total Days Scheduled column
df_annual = pd.read_excel(
    annual_schedule_path,
    sheet_name="Activity-Session Details",
    header=None
)

header_text = df_annual.apply(
    lambda col: col.astype("string").str.strip()
)

header_matches = (
    header_text.eq("Site").any(axis=1)
    & header_text.eq("Activity").any(axis=1)
    & header_text.eq("Session").any(axis=1)
    & header_text.eq("Days Scheduled").any(axis=1)
)

if not header_matches.any():
    raise ValueError("Cannot find the annual schedule headers.")

header_row = header_matches[header_matches].index[0]

df_annual.columns = header_text.iloc[header_row]
df_annual = df_annual.iloc[header_row + 1:].reset_index(drop=True)
df_annual.columns.name = None

merge_keys = ["Site", "Activity", "Session"]

df_annual = df_annual[
    merge_keys + ["Days Scheduled"]
].copy()

for col in merge_keys:
    df_annual[col] = df_annual[col].astype("string").str.strip()

# Remove blank-site and total rows
df_annual = df_annual.loc[
    df_annual["Site"].notna()
    & df_annual["Site"].ne("")
    & ~df_annual["Site"].str.lower().isin(
        ["total", "subtotal", "average"]
    )
].copy()

df_annual["Days Scheduled"] = pd.to_numeric(
    df_annual["Days Scheduled"], errors="coerce"
).astype("Int64")

# Remove exact duplicates, but stop on conflicting session counts
df_annual = df_annual.drop_duplicates()

duplicates = df_annual.duplicated(
    subset=merge_keys, keep=False
)

if duplicates.any():
    display(df_annual.loc[duplicates])
    raise ValueError("Conflicting annual schedule rows found.")

# Add yearly counts without replacing other session information
df_act = df_act.merge(
    df_annual,
    on=merge_keys,
    how="left",
    validate="many_to_one",
    indicator="_annual_match"
)

unmatched = df_act["_annual_match"].eq("left_only")

if unmatched.any():
    print("Sessions without a matching annual schedule:")
    display(df_act.loc[unmatched, merge_keys].drop_duplicates())

df_act = df_act.drop(columns="_annual_match")

display(
    df_act[
        merge_keys + ["Days Scheduled to Date", "Days Scheduled"]
    ].head(10)
)

/Users/lgteam/Library/Python/3.9/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Site,Activity,Session,Days Scheduled to Date,Days Scheduled
0,P.S. 181 Brooklyn,Arts & Crafts,Arts & Crafts 1,3,27
1,P.S. 181 Brooklyn,Arts & Crafts,Arts & Crafts 2,4,28
2,P.S. 181 Brooklyn,Arts & Crafts,Arts & Crafts K,3,27
3,P.S. 181 Brooklyn,Basketball,Basketball,7,55
4,P.S. 181 Brooklyn,Culture Club 4-8th/SEL,Culture Club/SEL,11,85
5,P.S. 181 Brooklyn,Dance,Dance/homework 1,4,31
6,P.S. 181 Brooklyn,Dance,Dance/homework 3,11,85
7,P.S. 181 Brooklyn,Dance,Dance/homework K,4,31
8,P.S. 181 Brooklyn,ELA,ELA Creators 3rd Grade,4,28
9,P.S. 181 Brooklyn,ELA,ELA Creators 4th Grade,4,28


In [28]:
# 6. Calculate the number of days missing attendance by site/activity/session/date
# Load Gary's Missing Attendance Report
missing_sheets = pd.read_excel(
    missing_attendance_path,
    sheet_name=None,
    header=None
)

missing_sheet_name = next(
    (
        name for name in missing_sheets
        if "Missing Attendance" in name
    ),
    None
)

if missing_sheet_name is None:
    raise ValueError("Missing Attendance report sheet not found.")

df_missing_att = missing_sheets[missing_sheet_name].copy()

# Detect the header row
header_text = df_missing_att.apply(
    lambda col: col.astype("string").str.strip()
)

header_matches = (
    header_text.eq("Site").any(axis=1)
    & header_text.eq("Activity").any(axis=1)
    & header_text.eq("Session").any(axis=1)
    & header_text.eq("Date").any(axis=1)
)

if not header_matches.any():
    raise ValueError("Cannot find Missing Attendance report headers.")

header_row = header_matches[header_matches].index[0]

df_missing_att.columns = header_text.iloc[header_row]
df_missing_att = (
    df_missing_att.iloc[header_row + 1:]
    .reset_index(drop=True)
)
df_missing_att.columns.name = None

merge_keys = ["Site", "Activity", "Session"]

df_missing_att = df_missing_att[merge_keys + ["Date"]].copy()

# Clean session identifiers
for col in merge_keys:
    df_missing_att[col] = (
        df_missing_att[col].astype("string").str.strip()
    )

# Remove blank-site and total rows
df_missing_att = df_missing_att.loc[
    df_missing_att["Site"].notna()
    & df_missing_att["Site"].ne("")
    & ~df_missing_att["Site"].str.lower().isin(
        ["total", "subtotal", "average"]
    )
].copy()

# Validate identifiers and dates
if df_missing_att[merge_keys].isna().any().any() or (
    df_missing_att[merge_keys].eq("").any().any()
):
    raise ValueError("Missing site, activity, or session in attendance report.")

df_missing_att["Date"] = pd.to_datetime(
    df_missing_att["Date"],
    errors="coerce"
).dt.normalize()

if df_missing_att["Date"].isna().any():
    raise ValueError("Invalid or missing dates in Missing Attendance report.")

# One missing day per site/activity/session/date
df_missing_att = (
    df_missing_att
    .drop_duplicates(subset=merge_keys + ["Date"])
    .sort_values(merge_keys + ["Date"])
)

# Convert dates to text after sorting
df_missing_att["Date Text"] = (
    df_missing_att["Date"].dt.strftime("%m/%d/%Y")
)

# Aggregate missing days and date lists by session
missing_attendance_summary = (
    df_missing_att.groupby(merge_keys)
    .agg(
        **{
            "Days Missing Attendance": ("Date", "nunique"),
            "Dates Missing Attendance": (
                "Date Text",
                lambda dates: "; ".join(dates)
            ),
        }
    )
    .reset_index()
)

display(missing_attendance_summary)

/Users/lgteam/Library/Python/3.9/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Site,Activity,Session,Days Missing Attendance,Dates Missing Attendance
0,P.S. 181 Brooklyn,Arts & Crafts,Arts & Crafts 2,1,10/30/2026
1,P.S. 181 Brooklyn,Basketball,Basketball,1,10/30/2026
2,P.S. 181 Brooklyn,Culture Club 4-8th/SEL,Culture Club/SEL,1,10/30/2026
3,P.S. 181 Brooklyn,Dance,Dance/homework 3,1,10/30/2026
4,P.S. 181 Brooklyn,Homework/tabletop games,Homework/Tabletop games 1,1,10/30/2026
5,P.S. 181 Brooklyn,Homework/tabletop games,Homework/Tabletop games K,1,10/30/2026


In [29]:
# 7. Create a site summary table for every grant site

sites = site_master["Site"].tolist()
merge_keys = ["Site", "Activity", "Session"]
site_tables = {}

# Check that missing-attendance sessions match existing session reports
known_sessions = pd.concat(
    [
        df_act[merge_keys],
        df_enr[merge_keys],
        df_att[merge_keys],
    ],
    ignore_index=True
).drop_duplicates()

unmatched_missing = missing_attendance_summary.merge(
    known_sessions,
    on=merge_keys,
    how="left",
    indicator=True
)

unmatched_missing = unmatched_missing.loc[
    unmatched_missing["_merge"].eq("left_only"),
    merge_keys
]

if not unmatched_missing.empty:
    print("Missing-attendance sessions not found in the session reports:")
    display(unmatched_missing)

    raise ValueError(
        "Check the institution, academic year, and session names "
        "before merging."
    )

output_columns = [
    "Site",
    "Activity",
    "Session",
    "Days Scheduled",
    "Enrolled Participant",
    "Average Daily Attendance",
    "Missing Attendance (%)",
    "Staff Assigned",
    "Partner Assigned",
    "Days Scheduled to Date",
    "Days Missing Attendance to Date",
    "Dates Missing Attendance",
    "Partner",
]

for site in sites:
    site_act = df_act.loc[df_act["Site"] == site].copy()
    site_enr = df_enr.loc[df_enr["Site"] == site].copy()
    site_att = df_att.loc[df_att["Site"] == site].copy()

    site_missing = missing_attendance_summary.loc[
        missing_attendance_summary["Site"] == site
    ].copy()

    # Merge session details, enrollment, and attendance
    merged_site = site_act.merge(
        site_enr,
        on=merge_keys,
        how="outer",
        validate="one_to_one"
    )

    merged_site = merged_site.merge(
        site_att,
        on=merge_keys,
        how="outer",
        validate="one_to_one"
    )

    # Add missing-attendance counts and dates
    merged_site = merged_site.merge(
        site_missing,
        on=merge_keys,
        how="left",
        validate="one_to_one"
    )

    # Exclude sessions starting today or later
    # Retain sessions with an unknown start date
    merged_site = merged_site.loc[
        merged_site["Session Start Date"].isna()
        | merged_site["Session Start Date"].lt(today)
    ].copy()

    # No row in Gary's report means no reported missing dates
    merged_site["Days Missing Attendance"] = (
        merged_site["Days Missing Attendance"]
        .fillna(0)
        .astype("Int64")
    )

    merged_site["Dates Missing Attendance"] = (
        merged_site["Dates Missing Attendance"].fillna("-")
    )

    for col in ["Staff Assigned", "Partner Assigned"]:
        merged_site[col] = merged_site[col].fillna("N")

    for col in ["Days Scheduled", "Days Scheduled to Date"]:
        merged_site[col] = pd.to_numeric(
            merged_site[col],
            errors="coerce"
        ).astype("Int64")

    # Calculate missing attendance using scheduled days to date
    # Zero or missing denominators produce "-"
    denominator = merged_site["Days Scheduled to Date"]

    missing_pct = (
        merged_site["Days Missing Attendance"]
        .div(denominator.where(denominator > 0))
        .mul(100)
    )

    # Display whole-number percentages, such as "25%"
    merged_site["Missing Attendance (%)"] = (
        missing_pct.round().astype("Int64").astype("string")
        + "%"
    ).fillna("-")

    # Flag inconsistent counts without capping the percentage
    inconsistent = (
        merged_site["Days Missing Attendance"].gt(
            merged_site["Days Scheduled to Date"]
        )
    ).fillna(False)

    if inconsistent.any():
        print(f"Check missing attendance counts for {site}:")
        display(
            merged_site.loc[
                inconsistent,
                merge_keys
                + ["Days Scheduled to Date", "Days Missing Attendance"]
            ]
        )

    # Rename the count for the final output
    merged_site = merged_site.rename(columns={
        "Days Missing Attendance": "Days Missing Attendance to Date"
    })

    # Select and order output columns
    merged_site = (
        merged_site[output_columns]
        .sort_values(["Activity", "Session"])
        .reset_index(drop=True)
    )

    # Replace missing output values safely
    merged_site = merged_site.astype(object).where(
        merged_site.notna(),
        "-"
    )

    site_tables[site] = merged_site

# Preview each site's table
for site_name, final_df in site_tables.items():
    print(f"\nSite: {site_name}")
    display(final_df.head())


Site: East New York Middle School of Excellence


,Site,Activity,Session,Days Scheduled,Enrolled Participant,Average Daily Attendance,Missing Attendance (%),Staff Assigned,Partner Assigned,Days Scheduled to Date,Days Missing Attendance to Date,Dates Missing Attendance,Partner



Site: P.S. 006 West Farms


,Site,Activity,Session,Days Scheduled,Enrolled Participant,Average Daily Attendance,Missing Attendance (%),Staff Assigned,Partner Assigned,Days Scheduled to Date,Days Missing Attendance to Date,Dates Missing Attendance,Partner



Site: P.S. 181 Brooklyn


,Site,Activity,Session,Days Scheduled,Enrolled Participant,Average Daily Attendance,Missing Attendance (%),Staff Assigned,Partner Assigned,Days Scheduled to Date,Days Missing Attendance to Date,Dates Missing Attendance,Partner
0,P.S. 181 Brooklyn,Arts & Crafts,Arts & Crafts 1,27,9,8,0%,N,N,3,0,-,-
1,P.S. 181 Brooklyn,Arts & Crafts,Arts & Crafts 2,28,16,15,25%,Y,Y,4,1,10/30/2026,CCM
2,P.S. 181 Brooklyn,Arts & Crafts,Arts & Crafts K,27,11,8,0%,N,N,3,0,-,-
3,P.S. 181 Brooklyn,Basketball,Basketball,55,30,21,14%,N,N,7,1,10/30/2026,-
4,P.S. 181 Brooklyn,Culture Club 4-8th/SEL,Culture Club/SEL,85,21,14,9%,Y,Y,11,1,10/30/2026,CCM


## 🌈 Color Coding and Full Report Download

In [30]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
from copy import copy


# Use the same sheet-name rules for export and highlighting
def clean_sheet_name(name):
    name = str(name).strip()

    for character in [":", "/", "\\", "?", "*", "[", "]"]:
        name = name.replace(character, "")

    return name.strip("'")[:31] or "Sheet"


def apply_missing_highlights(
    output_filename,
    total_missing_rows,
    flag_cols,
    young_dob_rows,
    staff_missing_rows=None,
    staff_flag_cols=None,
    site_tables=None
):
    red_fill = PatternFill(
        "solid", start_color="FF9999", end_color="FF9999"
    )
    blue_fill = PatternFill(
        "solid", start_color="9999FF", end_color="9999FF"
    )

    wb = load_workbook(output_filename)

    # Student required-information pull-outs
    flag_to_original = {
        flag: flag.removesuffix("_missing")
        for flag in flag_cols
        if flag.endswith("_missing")
        and flag != "State ParticipantID_missing"
    }

    for site, group in total_missing_rows.groupby("Site"):
        sheet_name = clean_sheet_name("Missing - " + str(site))
        ws = wb[sheet_name]
        headers = {cell.value: cell.column for cell in ws[1]}

        for row_idx, (_, row) in enumerate(group.iterrows(), start=2):
            for flag_col, original_col in flag_to_original.items():
                if original_col in headers and row.get(flag_col, 0) == 1:
                    ws.cell(
                        row=row_idx,
                        column=headers[original_col]
                    ).fill = red_fill

    # Staff required-information pull-out
    if staff_missing_rows is not None and staff_flag_cols is not None:
        ws = wb["Pull out - Missing Staff Info"]
        headers = {cell.value: cell.column for cell in ws[1]}

        staff_mapping = {
            "Staff Type_missing": "Staff Type",
            "Employment Type_missing": "Compensation Type",
            "Funded by 21st CCLC_missing": "Funder",
        }

        for row_idx, (_, row) in enumerate(
            staff_missing_rows.iterrows(), start=2
        ):
            for flag_col in staff_flag_cols:
                original_col = staff_mapping[flag_col]

                if original_col in headers and row.get(flag_col, 0) == 1:
                    ws.cell(
                        row=row_idx,
                        column=headers[original_col]
                    ).fill = red_fill

    # DOB/grade review: blue fill on DOB
    ws = wb["DOB - Atypical Age Check"]
    headers = {cell.value: cell.column for cell in ws[1]}

    if "Date Of Birth" in headers:
        for row_idx in range(2, ws.max_row + 1):
            ws.cell(
                row=row_idx,
                column=headers["Date Of Birth"]
            ).fill = blue_fill

    # Site session tables
    if site_tables is not None:
        for site_name in site_tables:
            ws = wb[clean_sheet_name(site_name)]
            headers = {
                cell.value: cell.column
                for cell in ws[1]
            }

            for row in ws.iter_rows(min_row=2):
                # Existing rules: blanks and unassigned staff/partners
                for cell in row:
                    column_name = ws.cell(
                        row=1, column=cell.column
                    ).value

                    is_blank = (
                        cell.value is None
                        or str(cell.value).strip() in ["", "-"]
                    )

                    assignment_missing = (
                        column_name in [
                            "Staff Assigned", "Partner Assigned"
                        ]
                        and str(cell.value).strip().upper() == "N"
                    )

                    if is_blank or assignment_missing:
                        cell.fill = red_fill

                # Highlight missing-attendance count, percentage, and dates
                count_col = "Days Missing Attendance to Date"

                if count_col in headers:
                    count_cell = ws.cell(
                        row=row[0].row,
                        column=headers[count_col]
                    )

                    try:
                        has_missing_attendance = float(count_cell.value) > 0
                    except (TypeError, ValueError):
                        has_missing_attendance = False

                    if has_missing_attendance:
                        for column_name in [
                            "Days Missing Attendance to Date",
                            "Missing Attendance (%)",
                            "Dates Missing Attendance",
                        ]:
                            if column_name in headers:
                                ws.cell(
                                    row=row[0].row,
                                    column=headers[column_name]
                                ).fill = red_fill
                            
    wb.save(output_filename)
    wb.close()

    print("Highlights applied.")

In [31]:
# Test output
from IPython.display import FileLink, display
import os

output_filename = "WeeklyUpdates_test_CCM_1030.xlsx"

# Hide internal checking columns
internal_columns = {
    "DOB_too_young",
    "DOB_age_grade_review",
    "_staff_id",
}

hide_cols = {
    "Race/Ethnicity",
    "English Learner Status",
    "Lunch Status",
    "Special Education Status",
    "IDEA Disability Type",
}

student_display_cols = [
    col for col in total_missing_rows.columns
    if not col.endswith("_missing")
    and col not in internal_columns
    and col not in hide_cols
]

dob_display_cols = [
    col for col in young_dob_rows.columns
    if not col.endswith("_missing")
    and col not in internal_columns
]

staff_display_cols = [
    col for col in staff_missing_rows.columns
    if not col.endswith("_missing")
    and col not in internal_columns
]

# Check for sheet-name collisions before writing
sheet_names = [
    "Student Summary Statistics",
    "Family Component Summary",
    "Missing Student Summary",
    "DOB - Atypical Age Check",
    "Missing Staff Summary",
    "Pull out - Missing Staff Info",
]

if not missing_site_rows.empty:
    sheet_names.append("Pull out - Missing Site Info")

sheet_names += [
    clean_sheet_name("Missing - " + str(site))
    for site in total_missing_rows["Site"].dropna().unique()
]

sheet_names += [
    clean_sheet_name(site)
    for site in site_tables
]

if len(sheet_names) != len(set(name.lower() for name in sheet_names)):
    raise ValueError(
        "Some output sheet names become identical after cleaning "
        "or truncation. Resolve the names before exporting."
    )

with pd.ExcelWriter(output_filename, engine="openpyxl") as writer:

    styled_totals.to_excel(
        writer,
        sheet_name="Student Summary Statistics",
        index=False
    )

    result.to_excel(
        writer,
        sheet_name="Family Component Summary",
        index=False
    )

    missing_summary.to_excel(
        writer,
        sheet_name="Missing Student Summary",
        index=False
    )

    # Keep these records visible if any students lack a site
    if not missing_site_rows.empty:
        missing_site_display_cols = [
            col for col in missing_site_rows.columns
            if not col.endswith("_missing")
            and col not in internal_columns
            and col not in hide_cols
        ]

        missing_site_rows[missing_site_display_cols].to_excel(
            writer,
            sheet_name="Pull out - Missing Site Info",
            index=False
        )

    # Required-information pull-out for each site with flagged students
    for site, group in total_missing_rows.groupby("Site"):
        group[student_display_cols].to_excel(
            writer,
            sheet_name=clean_sheet_name("Missing - " + str(site)),
            index=False
        )

    young_dob_rows[dob_display_cols].to_excel(
        writer,
        sheet_name="DOB - Atypical Age Check",
        index=False
    )

    staff_missing_summary.to_excel(
        writer,
        sheet_name="Missing Staff Summary",
        index=False
    )

    staff_missing_rows[staff_display_cols].to_excel(
        writer,
        sheet_name="Pull out - Missing Staff Info",
        index=False
    )

    # One session sheet for every master-list site
    for site_name, final_df in site_tables.items():
        final_df.to_excel(
            writer,
            sheet_name=clean_sheet_name(site_name),
            index=False
        )

# Apply formatting only after ExcelWriter has closed
apply_missing_highlights(
    output_filename,
    total_missing_rows,
    flag_cols,
    young_dob_rows,
    staff_missing_rows=staff_missing_rows,
    staff_flag_cols=staff_flag_cols,
    site_tables=site_tables
)

from copy import copy
from openpyxl import load_workbook

wb = load_workbook(output_filename)

print(f"Workbook saved to:\n{os.path.abspath(output_filename)}")

display(FileLink(output_filename))

Highlights applied.
Workbook saved to:
/Users/lgteam/Desktop/Weekly Updates Automation Tool/WeeklyUpdates_test_CCM_1030.xlsx


/Users/lgteam/Desktop/Weekly Updates Automation Tool/WeeklyUpdates_test_CCM_1030.xlsx